# **Lecture:** Tracking and video analysis

![](multi-object-tracking.jpg)

### **1. Learning Objectives**


By the end of this lab, you should be able to:

- Explain the difference between **detection** and **tracking**.
- Describe the role of **classical trackers** (KCF, CSRT) and when they are useful.
- Implement a simple **tracking-by-detection** pipeline using PyTorch and OpenCV.
- Run experiments on a **video** and observe how tracking behaves over time.
- Understand how tracking and detection relate to **real systems** like the DeepStream-based pedestrian analytics system described in the master’s thesis proposal (age and gender classification over detected people).


### **2. Detection vs Tracking**

#### 2.1 Object Detection

**Object detection** answers the question: *“What objects are in this frame, and where are they?”*.

- Input: a single image (frame).
- Output: bounding boxes + class labels (e.g., `person`).
- Typical models: Faster R-CNN, YOLO, SSD, etc.
- Cost: relatively **expensive** per frame (deep network forward pass).

#### 2.2 Object Tracking

**Object tracking** answers the question: *“Where did this object move between frames?”*.

- Input: an initial bounding box in frame *t*.
- Output: updated bounding box in frame *t+1, t+2, ...*.
- Trackers assume the object is the **same** and try to follow it over time.
- Cost: usually **cheaper** than running a full detector every frame.

Tracking is useful when:
- You want **smooth trajectories** of objects.
- You want to **reduce computation** by not running detection on every frame.
- You want to maintain **identity** (ID) of each person over time.

#### 2.3 Classical Trackers: KCF and CSRT

OpenCV provides several classical trackers, including:

- **KCF (Kernelized Correlation Filter)**
  - Fast, works well when the object appearance does not change too much.
  - Sensitive to occlusions and large scale changes.

- **CSRT (Discriminative Correlation Filter with Channel and Spatial Reliability)**
  - More accurate than KCF in many cases.
  - Slower than KCF, but still real-time on CPU for moderate resolutions.

These trackers are **not deep learning models**; they are classical computer vision algorithms.

#### 2.4 Tracking-by-Detection

**Tracking-by-detection** combines both ideas:

1. Run a **detector** every N frames (e.g., every 5 or 10 frames).
2. Initialize or update **trackers** for each detected object.
3. Between detection frames, use the trackers to **propagate** object positions.

Advantages:
- More robust than pure tracking (because detection can recover from drift).
- More efficient than running detection on every frame.
- Natural fit for systems that need **IDs over time**, like counting people entering a store or tracking their paths.

In the master’s thesis system, a similar idea is used: detect people, then classify age and gender for each detection, and aggregate statistics over time in a real deployment.

In [4]:
video_path = '/Users/eugenio/Documents/Computer_Vision/UC.06 Tracking & Video Análisis/race_car.mp4'  # Replace with your video path

In [1]:
from IPython.display import HTML
HTML("""
<video width="640" height="480" controls>
    <source src="/Users/eugenio/Documents/Computer_Vision/UC.06 Tracking & Video Análisis/race_car.mp4">
</video>
""")

### **KCF (Kernelized Correlation Filter)**
- Type: **Single Object Tracking (SOT)**
- Requires: **Initial bounding box only (first frame)**
- Does NOT use detection after initialization

**Key idea:** Track one object based only on its appearance from the first frame

In [ ]:
import cv2
from ultralytics import YOLO

# Cargar modelo
model = YOLO("yolo26n.pt")

# Leer video
cap = cv2.VideoCapture("race_car.mp4")
ret, frame = cap.read()

if not ret:
    raise Exception("No se pudo leer el video")

# Detectar SOLO en el primer frame
results = model(frame)

car_boxes = []
for box in results[0].boxes:
    if int(box.cls) == 2:
        car_boxes.append(box.xyxy.cpu().numpy())

if len(car_boxes) == 0:
    raise Exception("No se detectó ningún carro en el primer frame")

# Convertir bounding box
box = car_boxes[0].flatten()
x1, y1, x2, y2 = box
bbox = (int(x1), int(y1), int(x2 - x1), int(y2 - y1))

# Inicializar tracker
tracker = cv2.TrackerKCF_create()
tracker.init(frame, bbox)

# LOOP DE VISUALIZACIÓN
while True:
    ret, frame = cap.read()
    if not ret:
        break

    success, bbox = tracker.update(frame)

    if success:
        x, y, w, h = map(int, bbox)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0,255,0), 2)
        cv2.putText(frame, "Tracking", (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    else:
        cv2.putText(frame, "Lost", (50,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.imshow("KCF Tracking", frame)

    # ESC para salir
    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


0: 416x640 2 cars, 50.5ms
Speed: 11.3ms preprocess, 50.5ms inference, 2.3ms postprocess per image at shape (1, 3, 416, 640)


: 

### **CSRT (Discriminative Correlation Filter with Channel and Spatial Reliability)**
- Type: **Single Object Tracking (SOT)**
- Requires: **Initial bounding box only (first frame)**
- Does NOT use detection after initialization

**Key idea:** Improved version of KCF that adapts to scale and appearance changes

In [ ]:
import cv2
from ultralytics import YOLO

# Cargar modelo YOLO
model = YOLO("yolo26n.pt")

# Leer video
cap = cv2.VideoCapture("race_car.mp4")
ret, frame = cap.read()

if not ret:
    raise Exception("No se pudo leer el video")

# Detectar SOLO en el primer frame
results = model(frame)

car_boxes = []
for box in results[0].boxes:
    if int(box.cls) == 2:  # clase "car"
        car_boxes.append(box.xyxy.cpu().numpy())

if len(car_boxes) == 0:
    raise Exception("No se detectó ningún carro en el primer frame")

# Convertir bounding box (xyxy → xywh)
box = car_boxes[0].flatten()
x1, y1, x2, y2 = box
bbox = (int(x1), int(y1), int(x2 - x1), int(y2 - y1))

print("BBox inicial:", bbox)

# 🔥 Inicializar CSRT (más robusto que KCF)
tracker = cv2.TrackerCSRT_create()

# (si falla, usa esta alternativa)
# tracker = cv2.legacy.TrackerCSRT_create()

tracker.init(frame, bbox)

# LOOP de tracking
while True:
    ret, frame = cap.read()
    if not ret:
        break

    success, bbox = tracker.update(frame)

    if success:
        x, y, w, h = map(int, bbox)
        cv2.rectangle(frame, (x, y), (x+w, y+h), (255,0,0), 2)
        cv2.putText(frame, "CSRT Tracking", (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)
    else:
        cv2.putText(frame, "Lost", (50,50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.imshow("CSRT Tracking", frame)

    # ESC para salir
    if cv2.waitKey(30) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()


0: 416x640 2 cars, 24.3ms
Speed: 2.7ms preprocess, 24.3ms inference, 0.1ms postprocess per image at shape (1, 3, 416, 640)
BBox inicial: (1309, 412, 145, 101)


: 

### **YOLO + ByteTrack**
- Type: **Multi-Object Tracking (MOT)**
- Requires: **Detection on every frame (YOLO)**
- Uses IoU + confidence score for association

**Key idea:** Fast and efficient tracking-by-detection without deep appearance features

In [ ]:
from ultralytics import YOLO

# Load an official or custom model
model = YOLO("yolo26n.pt")  # Load an official Detect model

#results = model.track(source="/Users/eugenio/Documents/Computer_Vision/UC.06 Tracking & Video Análisis/race_car.mp4", show=True)  # Tracking with default tracker - BoT-SORT
results = model.track("race_car.mp4", show=True, tracker="bytetrack.yaml")  # with ByteTrack


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/210) /Users/eugenio/Documents/Computer_Vision/UC.06 Tracking & Video Análisis/race_car.mp4: 416x640 2 cars, 37.8ms
video 1/1 (frame 2/210) /Users/eugenio/Documents/Computer_Vision/UC.06 Tracking & Video Análisis/race_car.mp4: 416x640 2 cars, 25.2ms
video 1/1 (frame 3/210) /Users/eugenio/Documents/Computer_Vision/UC.06 Tracking & Video Análisis/race_car.mp4: 416x640 2 cars, 23.8ms
video 1/1 (frame 4/210) /Users/eugenio/Documents/Compute

: 

### **Summary**

| Method | Type | Uses Detection Every Frame? | # Objects | Robustness |
|-------|------|----------------------------|----------|-----------|
| KCF | SOT | ❌ No | 1 | Low |
| CSRT | SOT | ❌ No | 1 | Medium |
| YOLO + ByteTrack | MOT | ✅ Yes | Many | High |

**One-Line Intuition**

- KCF → "Follow this object fast"
- CSRT → "Follow this object carefully"
- YOLO + ByteTrack → "Track many objects efficiently"

### **The Tracking Challenge: Motion Trail + Heatmap + Smart Counter**

- Work in **groups of 2 students**

#### Objective

Build a **multi-feature tracking system** using a video of people (the video will be provided by the instructor) that includes:

1. Motion Trail (trajectory visualization)
2. Heatmap of activity
3. Smart Entry Counter

#### Task Overview

You will use an object tracking method (e.g., YOLO + ByteTrack) to:

- Track people across frames
- Extract their positions over time
- Build three different visual/analytical outputs

#### **Task 1: Motion Trail (Trajectory)**

Requirements:
- For each tracked person:
  - Compute the **center point** of the bounding box
  - Draw a **trail (path)** of previous positions
- The trail must:
  - Fade over time (older points disappear gradually)
  - Be updated frame-by-frame

#### **Task 2: Heatmap of Activity**

Requirements:
- Accumulate all detected center points into a spatial grid
- Generate a **heatmap visualization** showing:
  - High-density areas (more traffic)
  - Low-density areas
- Overlay the heatmap on the video or display separately

### **Task 3: Smart Entry Counter**

- Define a **virtual line** (e.g., entrance)
- Count how many people:
  - Cross the line in a specific direction
- Avoid double counting:
  - Each tracked ID should be counted only once

### **Deliverables**

Each group must submit:

1. Working code
2. Video output
3. Short explanation (ONLY ONE of the following per student) of the working logic:
   - Motion Trail
   - Heatmap
   - Counter logic

### **Grading Rubric (10 points total)**

#### Functionality (6 points)

| Component | Points | Criteria |
|----------|--------|---------|
| Motion Trail | 2 pts | Works correctly (center + fading) OR not |
| Heatmap | 2 pts | Correct accumulation and visualization OR not |
| Counter | 2 pts | Correct counting (no duplicates) OR not |

#### Explanation (4 points)

| Criteria | Points |
|--------|--------|
| Clear and correct explanation of chosen component | 4 pts |
| Incorrect or unclear explanation | 0 pts |
- No partial credit

---

<p style="text-align: right; font-size:14px; color:gray;">
<b>Prepared by:</b><br>
Manuel Eugenio Morocho-Cayamcela
</p>